# MAST Region Search Short Demo
Region search in MAST is supported via the [MAST CAOM TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/caom/).  We can use the `pyvo` package to make the TAP queries with the query constraints specified in SQL-like [Astronomical Data Query Language (ADQL)](https://www.ivoa.net/documents/ADQL/20231215/index.html).
## Imports

### Helper Functions
The ADQL for a region query is a bit involved, so we'll provide some helper functions to make the region part of the query easier to construct.  They'll start out in this notebook, but belong somewhere else like `MastAladin` or `astroquery`.

The user would typically call one of:

- `mast_reqion_query(region, other_clauses='', limit=50000)` 
- `mast_region_query_aladin_fov(aladin, other_clauses='', limit=50000, show_query=True)`

which would use the other two helper functions:
- `create_mast_region_adql(region, other_clauses='', limit=50000)`
- `create_adql_region(region)`

So far for demo purposes, `region` can be:
- an `s_region`
- list of points (RA/Dec tuples)
But it should be extended to support:
- astropy regions
- AIDA fovs

## Imports and Creating Aladin
Create Aladin in a sidecar centered on M101.

In [ ]:
from astropy import units as u
from astroquery.mast import Observations
from pyvo.dal import TAPService

from mast_aladin import AppSidecar, MastAladin

from mast_search_helpers import (
    create_adql_region, 
    create_mast_region_adql,
    mast_reqion_query,
    mast_region_query_aladin_fov
)

aladin = MastAladin(
    target="M101",
    fov=0.5
)
apps_initialized = AppSidecar.open(aladin, anchor='split-right')
AppSidecar.resize_all(675)

# Example: Search on Existing Footprint
Given the footprint of a JWST observation, search MAST for HST observations whose footprints overlap the JWST observation.
### Find and display a desired JWST footprint

In [ ]:
jwst_obs = Observations.query_criteria(obs_id='jw03429-o013_t005_nircam_clear-f444w')
jwst_obs_layer = aladin.add_table(jwst_obs, name='JWST Observation', color='green')

## Search on the Existing Footprint
Use the existing footprint as the `region` input for a MAST region TAP query.

This example adds the `other_clauses` argument to limit the results to HST.

Add the results as an Aladin overlay (in Portal footprint orange).

In [ ]:
# Search MAST for HST observations that overlap with the JWST observation.
footprint = jwst_obs[0]['s_region']
other_clauses = """
AND obs_collection = 'HST'
AND project = 'HST'
"""

hst_obs = mast_reqion_query(footprint, limit=50, other_clauses=other_clauses)
hst_obs_layer = aladin.add_table(hst_obs, name='HST Observations', color='#FF6600')

# Example: Search on Current Aladin FoV
Search on the polygon defined by Aladin's current viewport.  The polygon corners are computed using Aladin's notion of the WCS for the viewport.  We don't allow searches if the FoV is greater than 1 degree.

We'll use the existing Aladin instance.

## Search for HST on Current FoV.

In [ ]:
# Search MAST for HST observations that contain the current Aladin FoV.
other_clauses = """
AND obs_collection = 'HST'
AND project = 'HST'
"""
hst_fov_obs = mast_region_query_aladin_fov(aladin, limit=50, other_clauses=other_clauses)

# Add the results in yellow.
hst_fov_obs_layer = aladin.add_table(hst_fov_obs, name='HST FoV Observations', color='yellow')